# 使用脉冲化 Nature ResNet-18 进行 STAG 单帧分类

本 Notebook 是 `model_pro.py` 的训练、验证、可视化和模型保存入口。每个样本只包含一张 32×32 压力图；模型内部把同一张图重复为多个直接电流仿真步，这些仿真步不代表连续采集的多张触觉帧。

> 复现边界：2019 年 Nature STAG 论文实际使用普通 CNN。本项目仅保留论文描述的修改版 ResNet-18 空间拓扑，并把 ReLU 替换为 LIF 神经元，因此属于 SNN 工程改造版，不是原论文 SNN 的严格复现。

## 重要验证说明

按照当前实验协议，官方 `test` 划分会在每个 epoch 中作为 validation 使用，并参与选择 `best_model.pt`。这会产生模型选择偏差，最佳验证准确率不能作为无偏最终测试结果。

In [ ]:
from __future__ import annotations

import json
import os
import random
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score
from spikingjelly.activation_based import functional
from torch import nn
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# 同时兼容从仓库根目录和 code 目录启动 Notebook。
CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "data.py").is_file():
    CODE_DIR = CURRENT_DIR
elif (CURRENT_DIR / "code" / "data.py").is_file():
    CODE_DIR = CURRENT_DIR / "code"
else:
    raise FileNotFoundError("无法定位 code/data.py，请从仓库内启动 Notebook。")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
REPO_ROOT = CODE_DIR.parent

from data import create_single_frame_datasets
from model_pro import SpikingNatureResNet18, count_trainable_parameters

# 完整训练前改为 "full"，也可通过环境变量覆盖。
RUN_MODE = os.environ.get("MODEL_PRO_RUN_MODE", "smoke").strip().lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("MODEL_PRO_RUN_MODE 必须为 'smoke' 或 'full'。")

SEED = 42
LEARNING_RATE = 1e-3
GAUSSIAN_NOISE_STD = 0.015
DROPOUT = 0.2
TAU = 2.0
PROGRESS_UPDATE_INTERVAL = 10
CUDA_DEVICE_NAME: str | None = None
GPU_MEMORY_GIB: float | None = None

if RUN_MODE == "smoke":
    EPOCHS = 1
    BATCH_SIZE = 4
    VALIDATION_BATCH_SIZE = 4
    TIME_STEPS = 2
    SMOKE_TRAIN_SAMPLES = 4
    SMOKE_VALIDATION_SAMPLES = 4
    DEVICE = torch.device("cpu")
    NUM_WORKERS = 0
    PREFETCH_FACTOR = None
    PIN_MEMORY = False
    AMP_DTYPE = None
    EXPERIMENT_NAME = "model_pro_smoke"
else:
    EPOCHS = 200
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if DEVICE.type == "cuda":
        properties = torch.cuda.get_device_properties(0)
        CUDA_DEVICE_NAME = properties.name
        GPU_MEMORY_GIB = properties.total_memory / (1024 ** 3)
        if GPU_MEMORY_GIB >= 80:
            default_batch_size = 256
        elif GPU_MEMORY_GIB >= 40:
            default_batch_size = 128
        else:
            default_batch_size = 64
    else:
        default_batch_size = 64
    BATCH_SIZE = int(os.environ.get("MODEL_PRO_BATCH_SIZE", default_batch_size))
    VALIDATION_BATCH_SIZE = int(
        os.environ.get(
            "MODEL_PRO_VALIDATION_BATCH_SIZE",
            min(BATCH_SIZE * 2, 512),
        )
    )
    TIME_STEPS = int(os.environ.get("MODEL_PRO_TIME_STEPS", 16))
    default_workers = 0 if os.name == "nt" else min(
        8, max(1, (os.cpu_count() or 2) - 2)
    )
    NUM_WORKERS = int(
        os.environ.get("MODEL_PRO_NUM_WORKERS", default_workers)
    )
    PREFETCH_FACTOR = 4 if NUM_WORKERS > 0 else None
    PIN_MEMORY = DEVICE.type == "cuda"
    if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported():
        AMP_DTYPE = torch.bfloat16
    elif DEVICE.type == "cuda":
        AMP_DTYPE = torch.float16
    else:
        AMP_DTYPE = None
    EXPERIMENT_NAME = "model_pro_full"

if BATCH_SIZE <= 0 or VALIDATION_BATCH_SIZE <= 0:
    raise ValueError("批次大小必须为正整数。")
if TIME_STEPS <= 0 or NUM_WORKERS < 0:
    raise ValueError("时间步必须为正整数，worker 数量不能为负数。")

FUSED_ADAM = DEVICE.type == "cuda"
NON_BLOCKING = PIN_MEMORY
if DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

DATA_PATH = Path(
    os.environ.get(
        "STAG_DATA_PATH",
        REPO_ROOT / "stag_data" / "classification_lite.zip",
    )
).resolve()
OUTPUT_ROOT = Path(
    os.environ.get("MODEL_PRO_OUTPUT_ROOT", CODE_DIR / "outputs")
).resolve()
OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_WARNING = (
    "官方测试集在每个 epoch 中被用作验证集，并参与最佳模型选择；"
    "这会引入模型选择偏差，因此最佳验证分数不是无偏的最终测试估计。"
)
MODEL_DISCLOSURE = (
    "原始 Nature 论文使用普通 CNN；本模型是保留其修改版 ResNet-18 "
    "空间拓扑并使用 LIF 神经元的 SNN 工程改造版。"
)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id: int) -> None:
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

set_seed(SEED)
sns.set_theme(style="whitegrid")
print(f"运行模式：{RUN_MODE}")
print(f"计算设备：{DEVICE}")
if CUDA_DEVICE_NAME is not None:
    print(f"GPU：{CUDA_DEVICE_NAME}（{GPU_MEMORY_GIB:.1f} GiB）")
print(f"批次大小：{BATCH_SIZE}，验证批次：{VALIDATION_BATCH_SIZE}")
print(f"SNN 仿真时间步：{TIME_STEPS}")
print(f"数据加载进程：{NUM_WORKERS}，混合精度：{AMP_DTYPE}")
print(f"数据路径：{DATA_PATH}")
print(f"输出路径：{OUTPUT_DIR}")
print(f"模型说明：{MODEL_DISCLOSURE}")
print(f"验证警告：{VALIDATION_WARNING}")

## 1. 读取官方单帧数据划分

直接从 ZIP 中读取 `metadata.mat`，使用官方 `splitId` 与 `isBalanced`，并移除 `empty_hand`。Dataset 初始化时会一次性缓存归一化后的连续 `float32` 压力图。

In [ ]:
metadata, splits, full_train_dataset, full_validation_dataset = (
    create_single_frame_datasets(DATA_PATH)
)

assert metadata.num_frames == 135_187
assert int(metadata.sensor_mask.sum()) == 548
assert splits.num_classes == 26
assert len(full_train_dataset) == 35_178
assert len(full_validation_dataset) == 15_522
assert "empty_hand" not in splits.class_names
train_recordings = set(
    zip(
        metadata.batch_id[splits.train_indices].tolist(),
        metadata.recording_id[splits.train_indices].tolist(),
    )
)
validation_recordings = set(
    zip(
        metadata.batch_id[splits.validation_indices].tolist(),
        metadata.recording_id[splits.validation_indices].tolist(),
    )
)
assert train_recordings.isdisjoint(validation_recordings)

cache_mib = (
    full_train_dataset.cache_size_bytes
    + full_validation_dataset.cache_size_bytes
) / (1024 ** 2)
split_statistics = pd.DataFrame(
    {
        "split": ["Train", "Validation (Official Test)"],
        "samples": [len(full_train_dataset), len(full_validation_dataset)],
        "samples_per_class": [1_353, 597],
    }
)
display(split_statistics)
print(f"原始帧数：{metadata.num_frames:,}")
print(f"有效传感器数：{int(metadata.sensor_mask.sum())}")
print(f"物体类别数：{splits.num_classes}")
print(f"归一化数据缓存：{cache_mib:.2f} MiB")

## 2. 查看单帧压力图

图中的标题、坐标和颜色条统一使用英文。无效位置已经由 548 点传感器掩码清零。

In [ ]:
sample_indices = np.linspace(
    0, len(full_train_dataset) - 1, num=6, dtype=int
)
fig, axes = plt.subplots(2, 3, figsize=(10, 7), constrained_layout=True)
last_image = None
for axis, dataset_index in zip(axes.ravel(), sample_indices):
    image, label = full_train_dataset[int(dataset_index)]
    last_image = axis.imshow(image.squeeze(0), cmap="inferno", vmin=0, vmax=1)
    axis.set_title(f"Class: {splits.class_names[int(label)]}")
    axis.set_xlabel("Sensor column")
    axis.set_ylabel("Sensor row")
if last_image is not None:
    fig.colorbar(last_image, ax=axes, label="Normalized pressure", shrink=0.8)
fig.suptitle("STAG Single-Frame Pressure Samples", fontsize=15)
display(fig)
plt.close(fig)

## 3. 构建 DataLoader

完整模式在 Linux 上使用多进程预取、固定内存和持久 worker，以持续向 GPU 提供数据。冒烟模式仅抽取少量真实样本并固定在 CPU 执行。

In [ ]:
if RUN_MODE == "smoke":
    train_indices = np.linspace(
        0,
        len(full_train_dataset) - 1,
        num=SMOKE_TRAIN_SAMPLES,
        dtype=int,
    ).tolist()
    validation_indices = np.linspace(
        0,
        len(full_validation_dataset) - 1,
        num=SMOKE_VALIDATION_SAMPLES,
        dtype=int,
    ).tolist()
    train_dataset = Subset(full_train_dataset, train_indices)
    validation_dataset = Subset(
        full_validation_dataset, validation_indices
    )
else:
    train_dataset = full_train_dataset
    validation_dataset = full_validation_dataset

loader_options = {
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "persistent_workers": NUM_WORKERS > 0,
    "worker_init_fn": seed_worker,
}
if PREFETCH_FACTOR is not None:
    loader_options["prefetch_factor"] = PREFETCH_FACTOR

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=generator,
    **loader_options,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=VALIDATION_BATCH_SIZE,
    shuffle=False,
    **loader_options,
)

images, labels = next(iter(train_loader))
assert images.ndim == 4 and tuple(images.shape[1:]) == (1, 32, 32)
assert labels.ndim == 1
assert torch.isfinite(images).all()
assert images.min() >= 0 and images.max() <= 1
invalid_mask = torch.from_numpy(~metadata.sensor_mask)
assert torch.count_nonzero(images[:, 0, invalid_mask]) == 0
print(f"本次训练样本数：{len(train_dataset)}")
print(f"本次验证样本数：{len(validation_dataset)}")
print(f"输入批次形状：{tuple(images.shape)}")
print(
    f"DataLoader：workers={NUM_WORKERS}，pin_memory={PIN_MEMORY}，"
    f"prefetch_factor={PREFETCH_FACTOR}"
)

## 4. 创建脉冲化 Nature ResNet-18

模型保留修改版 ResNet-18 的前两个 layer group。输入通过直接电流编码重复为 `T` 个仿真步；LIF 状态在每个批次后重置。

In [ ]:
model = SpikingNatureResNet18(
    num_classes=splits.num_classes,
    time_steps=TIME_STEPS,
    tau=TAU,
    dropout=DROPOUT,
).to(DEVICE)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    fused=FUSED_ADAM,
)
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer, step_size=100, gamma=0.1
)
grad_scaler = torch.amp.GradScaler(
    "cuda", enabled=AMP_DTYPE == torch.float16
)
criterion = nn.CrossEntropyLoss()
sensor_mask_tensor = (
    torch.from_numpy(metadata.sensor_mask.astype(np.float32))
    .unsqueeze(0)
    .unsqueeze(0)
    .to(DEVICE)
)
performance_config = {
    "device": str(DEVICE),
    "cuda_device_name": CUDA_DEVICE_NAME,
    "gpu_memory_gib": GPU_MEMORY_GIB,
    "batch_size": BATCH_SIZE,
    "validation_batch_size": VALIDATION_BATCH_SIZE,
    "time_steps": TIME_STEPS,
    "num_workers": NUM_WORKERS,
    "prefetch_factor": PREFETCH_FACTOR,
    "pin_memory": PIN_MEMORY,
    "persistent_workers": NUM_WORKERS > 0,
    "amp_dtype": str(AMP_DTYPE) if AMP_DTYPE is not None else None,
    "tf32": DEVICE.type == "cuda",
    "cudnn_benchmark": DEVICE.type == "cuda",
    "fused_adam": FUSED_ADAM,
    "normalized_cache_mib": cache_mib,
}

model.eval()
try:
    with torch.no_grad():
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=AMP_DTYPE,
            enabled=AMP_DTYPE is not None,
        ):
            shape_check = model(
                images.to(DEVICE, non_blocking=NON_BLOCKING)
            )
finally:
    functional.reset_net(model)
assert tuple(shape_check.shape) == (len(labels), splits.num_classes)
assert torch.isfinite(shape_check).all()
assert model.encoder(images[:1]).shape == (
    TIME_STEPS, 1, 1, 32, 32
)
print(model)
print(f"可训练参数量：{count_trainable_parameters(model):,}")
print(f"模型输出形状：{tuple(shape_check.shape)}")
print(f"Fused Adam：{FUSED_ADAM}")

## 5. 定义训练与验证循环

指标先在 GPU 上累计，epoch 结束后再同步到 CPU。训练和验证都使用 `tqdm`；为了减少 GPU 同步，进度条中的即时 loss 每 10 个批次更新一次。

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    num_classes: int,
    description: str,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, object]:
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = torch.zeros((), device=device, dtype=torch.float32)
    top1_correct = torch.zeros((), device=device, dtype=torch.int64)
    top3_correct = torch.zeros((), device=device, dtype=torch.int64)
    sample_count = 0
    all_targets: list[torch.Tensor] = []
    all_predictions: list[torch.Tensor] = []
    epoch_start = time.perf_counter()

    progress_bar = tqdm(
        loader,
        desc=description,
        unit="批次",
        leave=False,
        dynamic_ncols=True,
        mininterval=0.5,
    )
    for batch_index, (batch_images, batch_labels) in enumerate(
        progress_bar, start=1
    ):
        batch_images = batch_images.to(
            device, non_blocking=NON_BLOCKING
        )
        batch_labels = batch_labels.to(
            device, non_blocking=NON_BLOCKING
        )

        if is_training:
            noise = torch.randn_like(batch_images) * GAUSSIAN_NOISE_STD
            batch_images = (batch_images + noise).clamp(0.0, 1.0)
            batch_images = batch_images * sensor_mask_tensor
            optimizer.zero_grad(set_to_none=True)

        try:
            with torch.set_grad_enabled(is_training):
                with torch.autocast(
                    device_type=device.type,
                    dtype=AMP_DTYPE,
                    enabled=AMP_DTYPE is not None,
                ):
                    logits = model(batch_images)
                    loss = criterion(logits, batch_labels)
                if is_training:
                    if grad_scaler.is_enabled():
                        grad_scaler.scale(loss).backward()
                        grad_scaler.step(optimizer)
                        grad_scaler.update()
                    else:
                        loss.backward()
                        optimizer.step()
        finally:
            functional.reset_net(model)

        batch_size = batch_labels.numel()
        sample_count += batch_size
        total_loss += loss.detach().float() * batch_size
        predictions = logits.detach().argmax(dim=1)
        top1_correct += (predictions == batch_labels).sum()
        top3 = logits.detach().topk(k=3, dim=1).indices
        top3_correct += (
            (top3 == batch_labels.unsqueeze(1)).any(dim=1).sum()
        )
        all_targets.append(batch_labels.detach())
        all_predictions.append(predictions)

        if (
            batch_index % PROGRESS_UPDATE_INTERVAL == 0
            or batch_index == len(loader)
        ):
            progress_bar.set_postfix(
                loss=f"{loss.detach().float().item():.4f}"
            )

    metric_totals = torch.stack(
        (total_loss, top1_correct.float(), top3_correct.float())
    ).cpu().numpy()
    targets = torch.cat(all_targets).cpu().numpy()
    predictions = torch.cat(all_predictions).cpu().numpy()
    elapsed_seconds = time.perf_counter() - epoch_start
    matrix = confusion_matrix(
        targets, predictions, labels=np.arange(num_classes)
    )
    macro_f1 = f1_score(
        targets,
        predictions,
        labels=np.arange(num_classes),
        average="macro",
        zero_division=0,
    )
    return {
        "loss": float(metric_totals[0]) / sample_count,
        "top1": float(metric_totals[1]) / sample_count,
        "top3": float(metric_totals[2]) / sample_count,
        "macro_f1": float(macro_f1),
        "sample_count": sample_count,
        "seconds": elapsed_seconds,
        "samples_per_second": sample_count / elapsed_seconds,
        "confusion_matrix": matrix,
    }


def scalar_metrics(metrics: dict[str, object]) -> dict[str, float | int]:
    return {
        "loss": float(metrics["loss"]),
        "top1": float(metrics["top1"]),
        "top3": float(metrics["top3"]),
        "macro_f1": float(metrics["macro_f1"]),
        "sample_count": int(metrics["sample_count"]),
        "seconds": float(metrics["seconds"]),
        "samples_per_second": float(metrics["samples_per_second"]),
    }


def checkpoint_payload(
    epoch: int,
    metrics: dict[str, object],
) -> dict[str, object]:
    return {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "model_config": model.get_config(),
        "performance_config": performance_config,
        "class_names": list(splits.class_names),
        "validation_metrics": scalar_metrics(metrics),
        "validation_uses_official_test": True,
        "validation_warning": VALIDATION_WARNING,
        "model_disclosure": MODEL_DISCLOSURE,
    }


def is_better_validation(
    metrics: dict[str, object],
    best_top1: float,
    best_loss: float,
) -> bool:
    top1 = float(metrics["top1"])
    loss = float(metrics["loss"])
    return top1 > best_top1 or (
        np.isclose(top1, best_top1, rtol=0.0, atol=1e-12)
        and loss < best_loss
    )

## 6. 执行训练与逐 epoch 验证

冒烟模式只运行少量真实样本的一次训练和验证；完整模式运行 200 个 epoch。学习率每 100 个 epoch 乘以 0.1。

In [ ]:
set_seed(SEED)
history: list[dict[str, float | int]] = []
best_epoch = -1
best_top1 = float("-inf")
best_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    current_learning_rate = float(optimizer.param_groups[0]["lr"])
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        DEVICE,
        splits.num_classes,
        description=f"训练 {epoch}/{EPOCHS}",
        optimizer=optimizer,
    )
    validation_metrics = run_epoch(
        model,
        validation_loader,
        criterion,
        DEVICE,
        splits.num_classes,
        description=f"验证 {epoch}/{EPOCHS}",
    )

    row: dict[str, float | int] = {
        "epoch": epoch,
        "learning_rate": current_learning_rate,
    }
    for prefix, metrics in (
        ("train", train_metrics),
        ("validation", validation_metrics),
    ):
        for name, value in scalar_metrics(metrics).items():
            row[f"{prefix}_{name}"] = value
    history.append(row)

    if is_better_validation(validation_metrics, best_top1, best_loss):
        best_epoch = epoch
        best_top1 = float(validation_metrics["top1"])
        best_loss = float(validation_metrics["loss"])
        torch.save(
            checkpoint_payload(epoch, validation_metrics),
            OUTPUT_DIR / "best_model.pt",
        )

    print(
        f"轮次 {epoch:03d}/{EPOCHS:03d} | "
        f"学习率={current_learning_rate:.2e} | "
        f"训练损失={train_metrics['loss']:.4f}，"
        f"Top-1={train_metrics['top1']:.3f} | "
        f"验证损失={validation_metrics['loss']:.4f}，"
        f"Top-1={validation_metrics['top1']:.3f}，"
        f"Top-3={validation_metrics['top3']:.3f} | "
        f"训练吞吐={train_metrics['samples_per_second']:.1f} 样本/秒，"
        f"验证吞吐={validation_metrics['samples_per_second']:.1f} 样本/秒"
    )
    scheduler.step()

torch.save(
    checkpoint_payload(EPOCHS, validation_metrics),
    OUTPUT_DIR / "last_model.pt",
)
history_frame = pd.DataFrame(history)
history_frame.to_csv(OUTPUT_DIR / "history.csv", index=False)
assert best_epoch >= 1
assert (OUTPUT_DIR / "best_model.pt").is_file()
assert (OUTPUT_DIR / "last_model.pt").is_file()
display(history_frame)

## 7. 复核最佳模型并绘制结果

重新载入最佳 checkpoint，在完整验证 DataLoader 上复核一次。所有保存图表只使用英文文本。

In [ ]:
best_checkpoint = torch.load(
    OUTPUT_DIR / "best_model.pt",
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_checkpoint["model_state_dict"])
set_seed(SEED)
best_validation_metrics = run_epoch(
    model,
    validation_loader,
    criterion,
    DEVICE,
    splits.num_classes,
    description="复核最佳模型",
)
print(f"选中的最佳轮次：{best_checkpoint['epoch']}")
print(f"重新计算的验证指标：{scalar_metrics(best_validation_metrics)}")
print(f"验证警告：{VALIDATION_WARNING}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
plots = (
    ("loss", "Cross-Entropy Loss", "Loss"),
    ("top1", "Top-1 Accuracy", "Accuracy"),
    ("top3", "Top-3 Accuracy", "Accuracy"),
    ("macro_f1", "Macro F1", "Score"),
)
for axis, (column, title, ylabel) in zip(axes.ravel(), plots):
    axis.plot(
        history_frame["epoch"],
        history_frame[f"train_{column}"],
        marker="o",
        label="Train",
    )
    axis.plot(
        history_frame["epoch"],
        history_frame[f"validation_{column}"],
        marker="o",
        label="Validation (Official Test)",
    )
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.set_ylabel(ylabel)
    axis.legend()
fig.suptitle("Spiking Nature ResNet-18 Training History", fontsize=15)
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
matrix = np.asarray(best_validation_metrics["confusion_matrix"])
fig, axis = plt.subplots(figsize=(14, 12), constrained_layout=True)
sns.heatmap(
    matrix,
    cmap="Blues",
    xticklabels=splits.class_names,
    yticklabels=splits.class_names,
    cbar_kws={"label": "Frame count"},
    ax=axis,
)
axis.set_title("Validation Confusion Matrix (Official Test Split)")
axis.set_xlabel("Predicted object")
axis.set_ylabel("True object")
axis.tick_params(axis="x", rotation=90)
axis.tick_params(axis="y", rotation=0)
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
row_totals = matrix.sum(axis=1)
class_accuracy = np.divide(
    np.diag(matrix),
    row_totals,
    out=np.zeros(splits.num_classes, dtype=np.float64),
    where=row_totals > 0,
)
fig, axis = plt.subplots(figsize=(14, 6), constrained_layout=True)
axis.bar(splits.class_names, class_accuracy, color="#4472C4")
axis.set_title("Per-Class Validation Accuracy (Official Test Split)")
axis.set_xlabel("Object class")
axis.set_ylabel("Accuracy")
axis.set_ylim(0.0, 1.0)
axis.tick_params(axis="x", rotation=90)
fig.savefig(OUTPUT_DIR / "class_accuracy.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

## 8. 保存类别映射和实验摘要

摘要同时保存模型复现边界、真实数据规模、性能配置和验证集选择偏差说明。

In [ ]:
class_mapping = [
    {
        "label": label,
        "original_object_id": int(original_id),
        "name": name,
    }
    for label, (original_id, name) in enumerate(
        zip(splits.original_object_ids, splits.class_names)
    )
]
(OUTPUT_DIR / "class_mapping.json").write_text(
    json.dumps(class_mapping, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

summary = {
    "run_mode": RUN_MODE,
    "num_frames": metadata.num_frames,
    "sensor_count": int(metadata.sensor_mask.sum()),
    "num_classes": splits.num_classes,
    "class_names": list(splits.class_names),
    "full_train_samples": len(full_train_dataset),
    "full_validation_samples": len(full_validation_dataset),
    "used_train_samples": len(train_dataset),
    "used_validation_samples": len(validation_dataset),
    "epochs": EPOCHS,
    "best_epoch": int(best_checkpoint["epoch"]),
    "selected_validation_metrics": best_checkpoint["validation_metrics"],
    "reevaluated_validation_metrics": scalar_metrics(
        best_validation_metrics
    ),
    "model_config": model.get_config(),
    "model_disclosure": MODEL_DISCLOSURE,
    "performance_config": performance_config,
    "optimizer": "Adam",
    "initial_learning_rate": LEARNING_RATE,
    "scheduler": {"name": "StepLR", "step_size": 100, "gamma": 0.1},
    "gaussian_noise_std": GAUSSIAN_NOISE_STD,
    "validation_uses_official_test": True,
    "validation_warning": VALIDATION_WARNING,
    "output_files": [
        "best_model.pt",
        "last_model.pt",
        "history.csv",
        "summary.json",
        "class_mapping.json",
        "training_curves.png",
        "confusion_matrix.png",
        "class_accuracy.png",
    ],
}
(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

expected_outputs = [OUTPUT_DIR / name for name in summary["output_files"]]
missing_outputs = [path.name for path in expected_outputs if not path.is_file()]
if missing_outputs:
    raise AssertionError(f"缺少输出文件：{missing_outputs}")

display(pd.DataFrame([summary["reevaluated_validation_metrics"]]))
print(f"实验结果已保存至：{OUTPUT_DIR}")
print(f"模型说明：{MODEL_DISCLOSURE}")
print(f"验证警告：{VALIDATION_WARNING}")

## 结果解释

本实验是对 Nature 修改版 ResNet-18 空间拓扑的脉冲化改造，使用单张压力图和直接电流编码。它适合与现有轻量 Conv-SNN 基线比较，但不能被称为原始 Nature SNN 复现。由于官方测试集参与逐 epoch 验证和最佳 checkpoint 选择，报告结果时必须表述为带有模型选择偏差的验证性能。